# הכנת דאטה אמיתי

המחברת הזו אינה מורידה אוטומטית קבצים בזמן פתיחה. היא נותנת פקודות מדויקות להרצה מקומית לאחר התקנת NCBI Datasets ו־MAFFT.

## 1. נתוני ייחוס

- HSV‑2 nucleotide reference: `NC_001798.2`
- Human reference: `GCF_000001405.40` — GRCh38.p14

מקורות רשמיים:
- https://www.ncbi.nlm.nih.gov/datasets/docs/v2/how-tos/virus/virus-download/
- https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_000001405.40/


In [ ]:
commands = r"""
# מתוך תיקיית הפרויקט
mkdir -p data/raw data/processed

# HSV-2 reference + annotations
datasets download virus genome accession NC_001798.2 \
  --include genome,cds,protein,annotation \
  --filename data/raw/hsv2_reference.zip

# אוסף גנומים מלאים של HSV-2
datasets download virus genome taxon "Human alphaherpesvirus 2" \
  --complete-only --include genome,annotation \
  --filename data/raw/hsv2_complete.zip

# GRCh38.p14
datasets download genome accession GCF_000001405.40 \
  --include genome,gff3 \
  --filename data/raw/human_GRCh38.zip
"""
print(commands)

## 2. אל תתחיל מכל הגנומים

ל־pilot קח 50–100 גנומים מלאים ואיכותיים:

- אורך קרוב לאורך ה־reference.
- מעט `N`.
- ללא כפילויות.
- metadata ממקורות שונים ככל האפשר.

שמור את רשימת ה־accessions המדויקת כדי שהניסוי יהיה reproducible.

## 3. יישור עם MAFFT

לאחר שבנית `data/processed/hsv2_sample_100.fasta`:


In [ ]:
print(r"""mafft --auto data/processed/hsv2_sample_100.fasta \
  > data/processed/hsv2_aligned.fasta""")

## 4. בדיקות איכות לפני חיפוש מטרות

- כל הרשומות באמת HSV‑2 ולא HSV‑1.
- אין רצפים קצרים או מקוטעים מאוד.
- ה־reference נמצא ב־alignment.
- אחוז gaps סביר.
- מזהי FASTA ייחודיים.
- ה־GFF מתאים לאותה גרסה בדיוק של גנום הייחוס.

אסור להשתמש במספרי מיקום מ־GFF של assembly אחר בלי לבצע mapping.

## 5. מה מריצים לאחר מכן?

החלף במחברת הדמו את הנתיבים:

```python
virus = read_fasta("data/processed/hsv2_aligned.fasta")
gff = read_gff3("data/raw/.../reference.gff3")
candidates = scan_spcas9_candidates(virus, reference_id="NC_001798.2", min_site_coverage=0.95)
```

לאחר מכן מייצאים את המועמדים ל־Cas-OFFinder במקום לסרוק את GRCh38 בפייתון.